# Phase10~13 Wide Visualization
Figure text is English only. Korean explanations are provided via print().


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid', context='talk')

DATA_DIR = Path('../data/phase10_13')
wide = pd.read_csv(DATA_DIR / 'wide_all_dedup.csv')
wphase = pd.read_csv(DATA_DIR / 'wide_phase_summary.csv')
waxis = pd.read_csv(DATA_DIR / 'wide_axis_summary.csv')
intent = pd.read_csv(DATA_DIR / 'intent_vs_observed_summary.csv')
router_scalar = pd.read_csv(DATA_DIR / 'router_stage_scalar.csv')
router_family = pd.read_csv(DATA_DIR / 'router_family_expert_long.csv')

print('데이터 로드 완료: wide rows =', len(wide), ', axis summary rows =', len(waxis), ', intent rows =', len(intent))
print('phase별 row 수:', wide.groupby('source_phase').size().to_dict())
print('diag 누락 run( wide ):', wide.loc[wide['diag_available'] != 1, 'run_phase'].tolist())


In [ ]:
print('Phase 단위 best/mean 성능 요약을 먼저 확인합니다. anchor 대비 delta를 함께 보면 phase별 난이도 차이를 빠르게 볼 수 있습니다.')

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
tmp = wphase.sort_values('source_phase').copy()

sns.barplot(data=tmp, x='source_phase', y='best_valid_mrr20', ax=axes[0], color='#4C78A8')
axes[0].set_title('Best Valid by Phase')
axes[0].set_xlabel('Phase')
axes[0].set_ylabel('Best Valid MRR@20')

sns.barplot(data=tmp, x='source_phase', y='mean_test_main', ax=axes[1], color='#F58518')
axes[1].set_title('Mean Test by Phase (Main Rows)')
axes[1].set_xlabel('Phase')
axes[1].set_ylabel('Mean Test MRR@20')

plt.tight_layout()
plt.show()


In [ ]:
print('phase별 axis group 내부에서 best valid를 비교합니다. 계획 문서의 axis 의도와 맞는 winner가 나오는지 첫 번째 체크포인트입니다.')

fig, axes = plt.subplots(2, 2, figsize=(16, 10.5))
phase_order = ['P10', 'P11', 'P12', 'P13']

for ax, phase in zip(axes.flat, phase_order):
    sub = waxis[waxis['source_phase'] == phase].sort_values('best_valid_mrr20', ascending=False)
    sns.barplot(data=sub, x='setting_group', y='best_valid_mrr20', ax=ax, palette='viridis')
    ax.set_title(f'{phase} Axis Best Valid')
    ax.set_xlabel('Axis Group')
    ax.set_ylabel('Best Valid MRR@20')
    ax.tick_params(axis='x', rotation=35)

plt.tight_layout()
plt.show()


In [ ]:
print('special slice 관점에서 cold / long-session이 test와 어떤 관계를 갖는지 봅니다. 일반 성능만 보면 놓치는 robustness 힌트를 얻기 위함입니다.')

main = wide[wide['n_completed'] >= 20].copy()

fig, axes = plt.subplots(1, 2, figsize=(13.8, 5.2))

sns.scatterplot(data=main, x='cold_item_mrr20', y='test_mrr20', hue='source_phase', style='source_phase', s=90, ax=axes[0])
axes[0].set_title('Cold Slice vs Test')
axes[0].set_xlabel('Cold Item MRR@20 (pop<=5)')
axes[0].set_ylabel('Test MRR@20')

sns.scatterplot(data=main, x='long_session_mrr20', y='test_mrr20', hue='source_phase', style='source_phase', s=90, ax=axes[1])
axes[1].set_title('Long Session vs Test')
axes[1].set_xlabel('Long Session MRR@20 (11+)')
axes[1].set_ylabel('Test MRR@20')

for ax in axes:
    ax.legend(loc='best', fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
print('diag와 성능의 관계를 확인합니다. top1 concentration/cv가 과도할 때 성능이 어떻게 움직이는지 phase별로 비교합니다.')

diag = main[main['diag_available'] == 1].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5.1))

sns.scatterplot(data=diag, x='diag_top1_max_frac', y='best_valid_mrr20', hue='source_phase', style='setting_group', s=90, ax=axes[0])
axes[0].set_title('Top1 Concentration vs Best Valid')
axes[0].set_xlabel('Diag Top1 Max Fraction')
axes[0].set_ylabel('Best Valid MRR@20')

sns.scatterplot(data=diag, x='diag_cv_usage', y='test_mrr20', hue='source_phase', style='setting_group', s=90, ax=axes[1])
axes[1].set_title('Usage CV vs Test')
axes[1].set_xlabel('Diag CV Usage')
axes[1].set_ylabel('Test MRR@20')

axes[0].legend(loc='best', fontsize=8)
axes[1].legend(loc='best', fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
print('plan intent 대비 관찰 delta를 heatmap으로 요약합니다. 양수(붉은색)는 anchor 대비 valid 개선, 음수(푸른색)는 하락입니다.')

pivot = intent.pivot_table(index='setting_group', columns='source_phase', values='delta_best_valid_vs_anchor', aggfunc='mean')

plt.figure(figsize=(9.5, 7.2))
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='coolwarm', center=0.0)
plt.title('Intent vs Observed: Delta Best Valid vs Anchor')
plt.xlabel('Phase')
plt.ylabel('Axis Group')
plt.tight_layout()
plt.show()

print('자동 매칭 실패 항목:')
print(intent.loc[intent['match_flag'] == 0, ['source_phase', 'setting_group', 'observed_tag', 'expected_pattern']].to_string(index=False))


In [ ]:
print('phase별 top 후보를 표로 확인합니다. wide 최종 shortlist를 만들 때 base가 되는 목록입니다.')

rank_cols = ['source_phase', 'setting_group', 'setting_key', 'best_valid_mrr20', 'test_mrr20', 'cold_item_mrr20', 'long_session_mrr20', 'diag_top1_max_frac', 'diag_cv_usage']
rank = (main.sort_values(['source_phase', 'best_valid_mrr20'], ascending=[True, False])
            .groupby('source_phase', as_index=False)
            .head(5)[rank_cols])
print(rank.to_string(index=False))


In [ ]:
print('router family-expert heatmap을 phase winner 기준으로 확인합니다. wide winner와 router 작동 패턴이 함께 맞는지 보는 셀입니다.')

winner_map = dict(zip(wphase['source_phase'], wphase['best_setting_key']))
winner_keys = [v for v in winner_map.values() if isinstance(v, str) and len(v) > 0]

sub = router_family[(router_family['setting_key'].isin(winner_keys)) & (router_family['stage_name'] == 'macro@1')].copy()
if sub.empty:
    print('router_family_expert_long에서 winner setting macro@1 row를 찾지 못했습니다.')
else:
    keep_experts = (sub.groupby('expert')['family_expert_share_norm'].mean().sort_values(ascending=False).head(8).index.tolist())
    sub = sub[sub['expert'].isin(keep_experts)]

    phases = ['P10', 'P11', 'P12', 'P13']
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    for ax, phase in zip(axes.flat, phases):
        key = winner_map.get(phase, None)
        one = sub[sub['setting_key'] == key]
        if one.empty:
            ax.set_title(f'{phase} winner heatmap (missing)')
            ax.axis('off')
            continue
        mat = one.pivot_table(index='family', columns='expert', values='family_expert_share_norm', aggfunc='mean').fillna(0.0)
        sns.heatmap(mat, cmap='magma', ax=ax)
        ax.set_title(f'{phase} Winner Family-Expert Heatmap')
        ax.set_xlabel('Expert')
        ax.set_ylabel('Family')

    plt.tight_layout()
    plt.show()


In [ ]:
print('마지막으로 winner들의 router scalar(top1/cv/n_eff)를 같이 비교합니다. 성능과 함께 collapse risk를 동시에 체크하기 위한 표입니다.')

winner_keys = set(wphase['best_setting_key'].tolist())
r = router_scalar[(router_scalar['setting_key'].isin(winner_keys)) & (router_scalar['stage_name'] == 'macro@1')].copy()

if r.empty:
    print('router_stage_scalar에서 winner row를 찾지 못했습니다.')
else:
    agg = (r.groupby(['source_phase', 'setting_key'], as_index=False)
             .agg(top1_max_frac=('top1_max_frac', 'mean'),
                  cv_usage=('cv_usage', 'mean'),
                  n_eff=('n_eff', 'mean'),
                  route_knn=('route_consistency_feature_group_knn_mean_score', 'mean')))
    print(agg.sort_values('source_phase').to_string(index=False))

    fig, ax = plt.subplots(figsize=(10.5, 5.2))
    sns.barplot(data=agg, x='setting_key', y='top1_max_frac', hue='source_phase', ax=ax)
    ax.set_title('Winner Top1 Concentration (macro@1)')
    ax.set_xlabel('Setting Key')
    ax.set_ylabel('Top1 Max Fraction')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()
